In [ ]:
import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg;
import pandas as pd


data=gutenberg.raw('shakespeare-hamlet.txt')

with open('hamlet.txt','w') as file:
    file.write(data)

In [ ]:
import numpy as np;
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [3]:
with open('hamlet.txt','r') as file:
    text=file.read().lower()

token=Tokenizer();
token.fit_on_texts([text]);
total_words=len(token.word_index)+1

In [4]:
total_words

4818

In [ ]:
input_sequnece=[];
for line in text.split('\n'):
    token_list=token.texts_to_sequences([line])[0]
    print(token_list)
    for i in range(1,len(token_list)):
        n_gram_sequence=token_list[:i+1]
        input_sequnece.append(n_gram_sequence)

In [6]:
input_sequnece[0:3]

[[1, 687], [1, 687, 4], [1, 687, 4, 45]]

In [7]:
max_seq_length=max([len(x) for x in input_sequnece])

In [8]:
max_seq_length

14

In [ ]:
input_sequnece

In [10]:
input_sequnece=np.array(pad_sequences(input_sequnece,maxlen=max_seq_length,padding='pre'))

In [11]:
input_sequnece[5]

array([   0,    0,    0,    0,    0,    0,    0,    1,  687,    4,   45,
         41, 1886, 1887])

In [12]:
import tensorflow as tf;
x,y=input_sequnece[:,:-1],input_sequnece[:,-1]



In [13]:
y

array([ 687,    4,   45, ..., 1047,    4,  193])

In [14]:
y=tf.keras.utils.to_categorical(y,num_classes=total_words)

In [15]:
y[1].shape

(4818,)

In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)

In [17]:
from tensorflow.keras.models import Sequential;
from tensorflow.keras.layers import Embedding,LSTM,Dropout,Dense;

In [ ]:
model=Sequential()
model.add(Embedding(total_words,100,input_length=max_seq_length-1))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

In [ ]:
history=model.fit(x_train,y_train,epochs=50,validation_data=(x_test,y_test),verbose=1)

In [25]:
text='I am very'
seq_text=token.texts_to_sequences([text])[0]
if len(seq_text)> max_seq_length:
    seq_text=seq_text[len(seq_text)-max_seq_length:]
padded_seq_text=pad_sequences([seq_text],maxlen=max_seq_length-1,padding='pre')
padded_seq_text

array([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  5, 92, 99]])

In [26]:
max_seq_length

14

In [27]:
y=model.predict(padded_seq_text)
predicted_word_index=np.argmax(y,axis=1)

1/1 [==============================] - 0s 261ms/step


In [28]:
predicted_word_index

array([1293], dtype=int64)

In [29]:
for word,index in token.word_index.items():
    if index==predicted_word_index:
        print(word)

deliuer'd


In [ ]:
model.save('lstm_gru.h5')

In [31]:
import pickle;

In [34]:
with open('token_pickle.pickle','wb') as file:
    pickle.dump(token,file,protocol=pickle.HIGHEST_PROTOCOL)

In [36]:
model.input_shape

(None, 13)